# Stage 4 — Descriptive & Unit Economics Analysis

Consumes committed Stage 3 processed evidence at `45568cc93be4fe09f460cb329c47f65977fd5f67`. Analysis is source-specific except four pre-authorized caveated comparisons. No project net earnings, realized deduction rate, fuel cost, inflation-adjusted primary values, or per-km earnings are constructed.

## 1. Validate Stage 3 and produce Stage 4 analytical outputs

Verify exact upstream blobs, preserve source definitions, create compact descriptive/comparison tables, and derive only two aligned SRC029 gross unit rates.

In [1]:

from pathlib import Path
from collections import defaultdict
import hashlib,os,pandas as pd
R=Path(os.environ['OJOL_REPO_DIR']); A=R/'data/analytical'; M=R/'metadata'; A.mkdir(parents=True,exist_ok=True)
S3='45568cc93be4fe09f460cb329c47f65977fd5f67'; B={'data/processed/driver_evidence_processed.csv':'f7b2743eff524c87d5510ab49aef080a8899672a','metadata/stage3_analysis_eligibility.csv':'606b4161c75540fc831bfe679b840d71c6437311','metadata/stage3_final_validation.csv':'990dbdb82d40e8eaa46a97ba8de5e3cfc03f9dd8','metadata/stage3_closure_summary.csv':'89ab9dce86fee95126b1eeab2516647754b305e3','metadata/stage3_closure_validation.csv':'44782d3822d993f899590a816db4100347f4bc94'}
def bs(p):
 b=p.read_bytes();return hashlib.sha1(f'blob {len(b)}\0'.encode()+b).hexdigest()
assert all((R/p).is_file() and bs(R/p)==s for p,s in B.items())
def rd(p):return pd.read_csv(R/p,dtype=str,keep_default_na=False)
p=rd('data/processed/driver_evidence_processed.csv');e=rd('metadata/stage3_analysis_eligibility.csv');fv=rd('metadata/stage3_final_validation.csv');cs=rd('metadata/stage3_closure_summary.csv');cv=rd('metadata/stage3_closure_validation.csv')
assert len(p)==72 and not p.observation_id.duplicated().any() and len(e)==17 and cs.iloc[0].closure_status=='PASS_WITH_CAVEAT' and not ((fv.status=='FAIL')&(fv.severity=='blocking')).any() and len(cv)==6 and (cv.status=='PASS').all()
assert set(e.loc[e.primary_analysis_eligibility=='not_eligible_for_primary_transformed_comparison','comparison_id'])=={'S2C001','S2C002','S2C005'} and not (e.stage2_comparability_status=='directly_comparable').any()
un=(p.denominator_status=='metric_specific_denominator_unresolved').sum();geo=' '.join(p.loc[p.source_id=='SRC013',['geography','extraction_notes']].agg(' '.join,axis=1));assert un==24 and '62' in geo and '67' in geo
L=defaultdict(list)
for _,x in e.iterrows():
 for i in filter(None,map(str.strip,x.observation_ids.split(';'))):L[i].append(f'{x.comparison_id}:{x.stage2_comparability_status}:{x.primary_analysis_eligibility}')
F={'earnings','deduction','operating_cost','activity','incentive'};d=p[p.metric_family.isin(F)&(p.observation_analysis_role!='context_evidence')].copy();d['stage4_treatment']=d.observation_analysis_role.map({'project_metric_evidence':'project_metric_source_specific','source_defined_descriptive_evidence':'source_defined_only'});d['comparison_links']=d.observation_id.map(lambda i:' | '.join(sorted(L.get(i,[]))))
d=d[['observation_id','source_id','metric_family','harmonized_metric','processed_value_numeric','processed_unit','statistic_type','category_label','denominator_status','temporal_evidence_status','stage4_treatment','comparison_links']].sort_values(['source_id','observation_id']);dp=A/'stage4_descriptive_results.csv';d.to_csv(dp,index=False,lineterminator='\n')
allow={'S2C003','S2C006','S2C007','S2C008'};ce=e[e.primary_analysis_eligibility=='eligible_with_caveat'];assert set(ce.comparison_id)==allow;bi=p.set_index('observation_id',drop=False);z=[]
for _,c in ce.iterrows():
 for i in filter(None,map(str.strip,c.observation_ids.split(';'))):
  o=bi.loc[i];z.append([c.comparison_id,c.comparison_use,c.stage2_comparability_status,o.source_id,i,o.processed_value_numeric,o.processed_unit,o.statistic_type,o.category_label,o.metric_denominator_n,o.denominator_status,o.observation_period_start,o.observation_period_end,o.geography])
cc=['comparison_id','comparison_use','comparability_status','source_id','observation_id','processed_value_numeric','processed_unit','statistic_type','category_label','metric_denominator_n','denominator_status','observation_period_start','observation_period_end','geography'];c=pd.DataFrame(z,columns=cc).sort_values(['comparison_id','source_id','observation_id']);cp=A/'stage4_comparison_results.csv';c.to_csv(cp,index=False,lineterminator='\n')
g,o,h=(bi.loc[i] for i in ['SRC029-O001','SRC029-O007','SRC029-O009']);K=['source_id','observation_period_start','observation_period_end','geography','metric_denominator_n'];assert all(all(g[k]==x[k] for k in K) for x in [o,h]) and [g.processed_unit,o.processed_unit,h.processed_unit]==['IDR/day','orders/day','hours/day'] and g.metric_denominator_n=='186' and h.working_time_basis=='source_reported_unspecified';gv,ov,hv=map(float,[g.processed_value_numeric,o.processed_value_numeric,h.processed_value_numeric])
u=pd.DataFrame([['S4UE001','gross_service_earnings_per_source_reported_working_hour','SRC029','SRC029-O001','SRC029-O009',gv,hv,round(gv/hv,2),'IDR/source-reported working hour'],['S4UE002','gross_service_earnings_per_completed_order','SRC029','SRC029-O001','SRC029-O007',gv,ov,round(gv/ov,2),'IDR/completed order']],columns=['unit_economics_id','metric_name','source_id','numerator_observation_id','denominator_observation_id','numerator_value','denominator_value','derived_value','unit']);u['value_provenance']='derived';u['derivation_basis']='ratio_of_source_means';u['metric_denominator_n']=186;u['observation_period']='2022-2023';u['geography']=g.geography;up=A/'stage4_unit_economics_results.csv';u.to_csv(up,index=False,lineterminator='\n')
print(f'Stage 3 verified: {S3[:12]} | outputs: {len(d)} descriptive, {c.comparison_id.nunique()} comparisons/{len(c)} rows, {len(u)} unit rates')


Stage 3 verified: 45568cc93be4 | outputs: 58 descriptive, 4 comparisons/25 rows, 2 unit rates


## 2. Methodological traceability, validation, and closure

Persist material decisions, test all blocking analytical restrictions, record caveats, and close the stage only when all expected outputs reproduce.

In [2]:

D=[('S4-MD001','Use exact committed Stage 3 evidence.','Consume validated upstream.','Stage 3 commit/blobs.','Preserve lineage.','Methodology — provenance'),('S4-MD002','Keep description source-specific.','No direct cross-source comparability.','Stage 3 eligibility.','No pooled estimate.','Methodology — comparability'),('S4-MD003','Preserve nominal values.','Primary CPI transforms deferred.','S2C001/002/005.','No primary real-value comparison.','Methodology — monetary'),('S4-MD004','Do not reconstruct project net.','Gross-to-net chain incomplete.','Stage 3 validation/S2C016.','Describe components only.','Methodology — reconstruction'),('S4-MD005','Treat reported deductions as prevalence.','Not matched transactions.','S2C008/017.','No realized rate.','Methodology — deductions'),('S4-MD006','Exclude mixed fuel-food bundles from project cost.','Includes personal spending.','S2C005/006.','No project-net subtraction.','Methodology — cost boundary'),('S4-MD007','Derive two SRC029 gross unit rates.','Aligned source means; n=186.','SRC029-O001/O007/O009.','Ratio-of-means only.','Results — unit economics'),('S4-MD008','Do not derive per-km earnings.','Distance basis unresolved.','S2C012.','Distance remains descriptive.','Methodology — distance'),('S4-MD009','Keep context evidence outside unit formulas.','Not realized economic quantities.','Stage 3 roles.','Interpretive context only.','Methodology — evidence roles')]
dec=pd.DataFrame(D,columns=['decision_id','decision','rationale','evidence_basis','analytical_implication','intended_report_destination']);mp=M/'stage4_methodological_decision_log.csv';dec.to_csv(mp,index=False,lineterminator='\n')
R4=[]
def q(i,n,ok,se='blocking',ev='',im='',cav=False):R4.append([i,n,'CAVEAT' if ok and cav else 'PASS' if ok else 'FAIL',se,ev,im])
trans=((p.processed_transformation_status!='none')|(p.processed_transformation_formula.str.strip()!='')).sum();q('S4V001','Stage 3 blobs verified',True,ev='5 exact blobs');q('S4V002','Stage 3 closure valid',True,ev='PASS_WITH_CAVEAT');q('S4V003','Input counts preserved',len(p)==72 and len(e)==17,ev='72/17');q('S4V004','Observation IDs unique',not p.observation_id.duplicated().any());q('S4V005','No primary transformed input',trans==0,ev=f'{trans} rows');q('S4V006','Descriptive evidence complete',len(d)==58,ev=f'{len(d)} rows');q('S4V007','Only four caveated comparisons',set(c.comparison_id)==allow and set(c.comparability_status)=={'comparable_with_caveat'},ev=str(sorted(allow)));q('S4V008','Deferred CPI comparisons absent',not(set(c.comparison_id)&{'S2C001','S2C002','S2C005'}));bad=set(e.loc[e.primary_analysis_eligibility.isin(['not_comparable','context_only']),'comparison_id']);q('S4V009','Rejected/context uses absent',not(set(c.comparison_id)&bad));E={'gross_service_earnings_per_source_reported_working_hour','gross_service_earnings_per_completed_order'};q('S4V010','Only two aligned SRC029 gross rates',set(u.metric_name)==E and len(u)==2);q('S4V011','Hourly basis remains generic',h.working_time_basis=='source_reported_unspecified');q('S4V012','No per-km rate',not u.metric_name.str.contains('per_km',regex=False).any());bn=[x for x in u.metric_name if any(t in x for t in ['net_operating','driver_receipts','realized_driver_deduction','fuel_cost'])];q('S4V013','No project net/receipts/realized deduction/fuel',not bn);I=set(u.numerator_observation_id)|set(u.denominator_observation_id);q('S4V014','No mixed-cost formula input',not(I&set(p.loc[p.cost_boundary=='mixed_work_personal','observation_id'])));q('S4V015','No reported-deduction formula input',not(I&set(p.loc[p.semantic_flags.str.contains('driver_reported_deduction_not_realized',regex=False),'observation_id'])));q('S4V016','No source-net formula input',not(I&set(p.loc[p.earnings_layer=='source_defined_net','observation_id'])));q('S4V017','Nine decisions traceable',len(dec)==9 and not dec.eq('').any().any());q('S4V018','Unresolved denominators explicit',un==24,'non_blocking',str(un),'Caveat respondent counts.',True);q('S4V019','SRC013 locality discrepancy explicit','62' in geo and '67' in geo,'non_blocking','62/67','Do not assert one count.',True);q('S4V020','No directly comparable cross-source use',not(e.stage2_comparability_status=='directly_comparable').any(),'non_blocking','0 direct','Keep caveats.',True);chain={'driver_gross_service_earnings','driver_side_platform_deduction','driver_receipts_before_operating_cost','fuel_cost'};miss=sorted(chain-set(p.harmonized_metric.replace('',pd.NA).dropna()));q('S4V021','Gross-to-net chain incomplete',bool(miss),'non_blocking',str(miss),'Do not claim project net.',True);q('S4V022','Rates are ratios of means',set(u.derivation_basis)=={'ratio_of_source_means'},'non_blocking','2 rates','Not mean driver ratios.',True)
v=pd.DataFrame(R4,columns=['check_id','check_name','status','severity','evidence','analytical_implication']);vp=M/'stage4_analysis_validation.csv';v.to_csv(vp,index=False,lineterminator='\n');f=v[(v.status=='FAIL')&(v.severity=='blocking')];assert f.empty
spec=[('S4-OUT001',dp,'source_specific_descriptive_results',len(d),'Source-specific descriptive evidence.'),('S4-OUT002',cp,'caveated_comparison_source_values',len(c),'Four caveated comparison uses.'),('S4-OUT003',up,'derived_unit_economics_results',len(u),'Two aligned SRC029 gross rates.'),('S4-OUT004',mp,'methodological_decision_log',len(dec),'Material decisions.'),('S4-OUT005',vp,'analysis_validation',len(v),'Stage 4 validation.')];man=pd.DataFrame([[i,str(x.relative_to(R)),t,n,r] for i,x,t,n,r in spec],columns=['output_id','output_path','output_type','row_count','role']);manp=M/'stage4_output_manifest.csv';man.to_csv(manp,index=False,lineterminator='\n');me=[x.output_id for _,x in man.iterrows() if not (R/x.output_path).is_file() or len(pd.read_csv(R/x.output_path,dtype=str,keep_default_na=False))!=int(x.row_count)];assert not me
status='PASS_WITH_CAVEAT' if (v.status=='CAVEAT').any() else 'PASS';s=pd.DataFrame([['Stage 4','Descriptive & Unit Economics Analysis',status,S3,len(p),len(d),c.comparison_id.nunique(),len(c),len(u),len(dec),len(v),(v.status=='PASS').sum(),(v.status=='CAVEAT').sum(),(v.status=='FAIL').sum(),len(f),'Source-specific analysis; four caveated comparisons; two aligned SRC029 gross rates; no project net/realized deduction/fuel/per-km rate.']],columns=['stage','stage_title','closure_status','stage3_input_commit','processed_input_observation_count','descriptive_result_row_count','caveated_comparison_use_count','caveated_comparison_source_value_row_count','derived_unit_economics_result_count','methodological_decision_count','validation_check_count','validation_pass_count','validation_caveat_count','validation_fail_count','blocking_failure_count','key_conclusion']);sp=M/'stage4_closure_summary.csv';s.to_csv(sp,index=False,lineterminator='\n')
C=[('S4CL001','No blocking validation failure',f.empty),('S4CL002','Manifest reproduces outputs',not me),('S4CL003','Only authorized numeric comparisons',set(c.comparison_id)==allow),('S4CL004','Only two SRC029 gross rates',set(u.metric_name)==E and len(u)==2),('S4CL005','No prohibited unit rate',not bn and not u.metric_name.str.contains('per_km',regex=False).any()),('S4CL006','Decisions traceable',len(dec)==9 and not dec.eq('').any().any())];cl=pd.DataFrame([[i,n,'PASS' if ok else 'FAIL','blocking',str(ok),'Closure gate.'] for i,n,ok in C],columns=['check_id','check_name','status','severity','evidence','analytical_implication']);clp=M/'stage4_closure_validation.csv';cl.to_csv(clp,index=False,lineterminator='\n');assert (cl.status=='PASS').all()
for x in [dp,cp,up,mp,vp,manp,sp,clp]:assert x.is_file()
print(f'Stage 4: {status} | validation {(v.status=="PASS").sum()} PASS/{(v.status=="CAVEAT").sum()} CAVEAT/{(v.status=="FAIL").sum()} FAIL | closure {(cl.status=="PASS").sum()}/6')


Stage 4: PASS_WITH_CAVEAT | validation 17 PASS/5 CAVEAT/0 FAIL | closure 6/6
